# Mock de MLflow: experimento y modelo sin entrenar

Este notebook **no entrena ningún modelo**. Sirve para comprobar la conexión con MLflow:

1. Se conecta al servidor de tracking.
2. Registra un experimento con un run y métricas fijas.
3. Registra un modelo falso en el Model Registry y lo carga de nuevo.

Requisito: MLflow levantado (ver `services/README.md`), en `http://localhost:5000`.

## 1. Conexión con MLflow

In [1]:
import os

import mlflow
from mlflow import MlflowClient

TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimentos existentes:", [e.name for e in client.search_experiments()])

c:\Users\Henry\.venvs\mlops-churn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tracking URI: http://localhost:5000
Experimentos existentes: ['mock-churn', 'churn-mlops', 'Default']


## 2. Experimento y run con métricas fijas

In [2]:
EXPERIMENTO = "mock-churn"
mlflow.set_experiment(EXPERIMENTO)

with mlflow.start_run(run_name="mock-run") as run:
    mlflow.log_params({"modelo": "mock", "entrenado": False})
    mlflow.log_metrics({"auc": 0.75, "recall": 0.60})
    run_id = run.info.run_id

print("run_id:", run_id)

🏃 View run mock-run at: http://localhost:5000/#/experiments/2/runs/306830a8d8c948268d0cbdacfe62039d
🧪 View experiment at: http://localhost:5000/#/experiments/2
run_id: 306830a8d8c948268d0cbdacfe62039d


## 3. Modelo falso

`ModeloMock` no aprende nada: devuelve siempre la misma probabilidad de churn. Basta para probar el registro y la carga desde MLflow.

In [3]:
import numpy as np
from mlflow.pyfunc import PythonModel


class ModeloMock(PythonModel):
    def __init__(self, probabilidad=0.3):
        self.probabilidad = probabilidad

    def predict(self, context, model_input, params=None):
        return np.full(len(model_input), self.probabilidad)

c:\Users\Henry\.venvs\mlops-churn\Lib\site-packages\mlflow\pyfunc\utils\data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


## 4. Registrar el modelo en el Model Registry

In [4]:
NOMBRE_MODELO = "churn-mock"

with mlflow.start_run(run_id=run_id):
    info = mlflow.pyfunc.log_model(
        name="modelo",
        python_model=ModeloMock(),
        registered_model_name=NOMBRE_MODELO,
    )

print("model_uri:", info.model_uri)

2026/10/12 08:50:04 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
Registered model 'churn-mock' already exists. Creating a new version of this model...
2026/10/12 08:50:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: churn-mock, version 2
Created version '2' of model 'churn-mock'.


🏃 View run mock-run at: http://localhost:5000/#/experiments/2/runs/306830a8d8c948268d0cbdacfe62039d
🧪 View experiment at: http://localhost:5000/#/experiments/2
model_uri: models:/m-8ed0d180113e428eb6a5a539f8afb9a3


## 5. Comprobar el modelo registrado

In [5]:
import pandas as pd

versiones = client.search_model_versions(f"name='{NOMBRE_MODELO}'")
ultima = max(versiones, key=lambda v: int(v.version))
print("Versión:", ultima.version, "| estado:", ultima.status)

modelo = mlflow.pyfunc.load_model(f"models:/{NOMBRE_MODELO}/{ultima.version}")
print("Predicciones:", modelo.predict(pd.DataFrame({"x": [1, 2, 3]})))

Versión: 2 | estado: READY


Predicciones: [0.3 0.3 0.3]


## 6. Dónde verlo

En la UI de MLflow (http://localhost:5000):

- Experimento `mock-churn`, con el run `mock-run` y sus métricas.
- Modelo `churn-mock` en el Model Registry, con la versión creada aquí.